In [1]:
import sys
from pathlib import Path
import numpy as np
import torch
import pandas as pd
from chronos import Chronos2Pipeline

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    backtest_21d_rolling,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_CONTEXT_CHRONOS,
    ARTIFACTS_DIR,
    TICKERS,
)

MODEL_ID = "amazon/chronos-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

c:\capstone_project_unfc\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(f"Loading {MODEL_ID} on {DEVICE}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.float32,
)

def get_forecast_chronos(context: pd.Series, horizon: int):
    """Return list of `horizon` point forecasts (0.5 quantile) from Chronos."""
    try:
        # Chronos requires regular frequency (no gaps). Daily equity data has weekends/holidays,
        # so resample to business-day range and ffill so pd.infer_freq() succeeds.
        context = context.astype(float)
        idx = pd.date_range(context.index.min(), context.index.max(), freq="B")
        context_regular = context.reindex(idx).ffill().bfill().dropna()
        if len(context_regular) < 3:
            return []
        target_values = context_regular.values.flatten()
        timestamps = context_regular.index
        input_df = pd.DataFrame({
            "item_id": ["x"] * len(target_values),
            "timestamp": timestamps,
            "target": target_values,
        })
        forecast_df = pipeline.predict_df(
            input_df,
            prediction_length=horizon,
            quantile_levels=[0.5],
            id_column="item_id",
            timestamp_column="timestamp",
            target="target",
            validate_inputs=True,
        )
        qcol = "0.5" if "0.5" in forecast_df.columns else "predictions"
        if qcol not in forecast_df.columns:
            qcol = forecast_df.columns[-1]
        q = forecast_df[qcol]
        vals = np.asarray(q).ravel()
        return [float(vals[i]) for i in range(min(horizon, len(vals)))]
    except Exception as e:
        print(f"Chronos forecast error: {e}")
        return []

Loading amazon/chronos-2 on cpu...


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
stacked = load_pool_data()
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close
0,2021-03-09,AAPL,121.089996
1,2021-03-10,AAPL,119.980003
2,2021-03-11,AAPL,121.959999
3,2021-03-12,AAPL,121.029999
4,2021-03-15,AAPL,123.989998
5,2021-03-16,AAPL,125.570000
6,2021-03-17,AAPL,124.760002
7,2021-03-18,AAPL,120.529999
8,2021-03-19,AAPL,119.989998
9,2021-03-22,AAPL,123.389999


In [4]:
# Diagnostic: run one Chronos forecast and inspect output (delete or skip after debugging)
_sym = TICKERS[0]
_grp = stacked[stacked["symbol"] == _sym]
_prices = _grp.set_index("timestamp")["close"].astype(float).dropna().sort_index()
_context = _prices.iloc[-200:]  # last 200 points
_input_df = pd.DataFrame({
    "item_id": ["x"] * len(_context),
    "timestamp": _context.index,
    "target": _context.values,
})
print("Input shape:", _input_df.shape, "| timestamp dtype:", _input_df["timestamp"].dtype)
try:
    _out = pipeline.predict_df(_input_df, prediction_length=21, quantile_levels=[0.5],
                               id_column="item_id", timestamp_column="timestamp", target="target")
    print("Forecast shape:", _out.shape, "| columns:", list(_out.columns))
    print(_out.head(3))
    _got = get_forecast_chronos(_context, 21)
    print("get_forecast_chronos returned len:", len(_got), "| first 3:", _got[:3] if _got else "[]")
except Exception as e:
    print("Error:", type(e).__name__, e)

Input shape: (200, 3) | timestamp dtype: datetime64[s]
Error: ValueError Could not infer frequency for series x


In [5]:
# 21-day-ahead direct forecast; 60-day test window, rolling by 7 days (held-out test assets only)
model_name = "chronos"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym]
    if grp.empty:
        continue
    prices = grp.set_index("timestamp")["close"].astype(float).dropna().sort_index()
    if len(prices) < TEST_SIZE + MIN_CONTEXT_CHRONOS:
        continue
    pred = backtest_21d_rolling(
        prices, TEST_SIZE, FORECAST_HORIZON, ROLLING_STEP,
        MIN_CONTEXT_CHRONOS,
        get_forecast_fn=get_forecast_chronos,
    )
    if pred.empty:
        continue
    pred["symbol"] = sym
    all_preds.append(pred)

pred_chronos = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_chronos.groupby("symbol").size() if not pred_chronos.empty else "No predictions (all test symbols skipped or backtest returned empty).")
pred_chronos.head()

c:\capstone_project_unfc\env\Lib\site-packages\chronos\chronos2\dataset.py:89: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  task_target = torch.from_numpy(task_target)


symbol
AAPL     126
AMZN     126
GOOGL    126
JNJ      126
JPM      126
MSFT     126
NVDA     126
SPY      126
WMT      126
XOM      126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-10,278.779999,277.575928,0,AAPL
1,2025-12-11,278.029999,277.771820,0,AAPL
2,2025-12-12,278.279999,278.133911,0,AAPL
3,2025-12-15,274.109985,277.992126,0,AAPL
4,2025-12-16,274.609985,278.153137,0,AAPL


In [6]:
metrics_rows = []
for sym in pred_chronos["symbol"].unique():
    sub = pred_chronos[pred_chronos["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_chronos)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_chronos_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_chronos_pool.parquet")

      model   symbol        MAE       RMSE    MAPE_%
0   chronos     AAPL  11.147629  13.244213  4.235835
1   chronos     MSFT  24.446923  28.355823  5.749320
2   chronos    GOOGL  13.029338  14.786100  4.065563
3   chronos     AMZN  13.398862  15.886446  6.141126
4   chronos      JPM  12.703494  14.151888  4.050278
5   chronos      JNJ  12.451304  13.916480  5.425908
6   chronos      WMT   4.897155   5.577521  3.994677
7   chronos      SPY   5.933265   7.075163  0.863815
8   chronos      XOM   7.773772   9.161092  5.578577
9   chronos     NVDA   6.244659   7.320721  3.397057
10  chronos  overall  11.202640  15.065149  4.350216
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_chronos_pool.parquet
